## Transducers from Clojure

What we call transduscers is a concept formulated in the language Clojure. Transduscers are refered to as "composable algorithm transformations".
Another more useful definition given by Erik Assum is that "transduscers are transformations of reducing functions".
Loosely it says:
```
((input, whatever) -> whatever) -> ((input,whatever) -> whatever)
```

In the example above, the reducing functions are `(input, whatever) -> whatever`, i.e. a function that takes an `input` and a
`whatever` and returns a `whatever`. 

Our goal is to formalize this possibily with Category Theory. We need to figure out
what are the reducing functions and then figure out what are the transducers.

In [1]:
using Pkg
Pkg.activate(".")

  Activating project at `~/Documents/GitHub/Julia_Tutorials/FunctionalProgramming`


We start from the `map` function. Our goal to write such function using `foldl`.

In [2]:
map(x->x+1, [1,2,3])

3-element Vector{Int64}:
 2
 3
 4

In [3]:
inc(x) = x + 1 
foldl(
    (acc,s)->vcat(acc,inc(s)),
    [1,2,3];
    init=[]
)

3-element Vector{Any}:
 2
 3
 4

Our code works. We can make it cleaner. 

In [4]:
function map_inc_vcat(acc, s)
    vcat(acc,inc(s))
end

foldl(map_inc_vcat, [1,2,3], init=[])

3-element Vector{Any}:
 2
 3
 4

Let us make it even more general.

In [6]:
function map_inc(op,acc, s)
    op(acc,inc(s))
end
# curried version
function map_inc(op)
    (acc,s) -> op(acc,inc(s))
end
foldl(map_inc(vcat), [1,2,3], init=[])

3-element Vector{Any}:
 2
 3
 4

Abstracting even more.

In [7]:
function map_(op,f, acc, s)
    op(acc,f(s))
end
# curried version
function map_(f,op)
    (acc,s) -> op(acc,f(s))
end

# completely curried version
function map_(f)
    op-> begin
        (acc,s) -> op(acc,f(s))
    end
end
foldl(map_(inc, vcat), [1,2,3], init=[])

3-element Vector{Any}:
 2
 3
 4

In [8]:
foldl(map_(inc)(vcat), [1,2,3], init=[])

3-element Vector{Any}:
 2
 3
 4

In this context, the reducing function is `op` . Our function `map_inc` takes a  reducing function
and returns a reducing function. Moreover, our `map_` function takes
a function `f` and a reducing function `op` and returns a reducing function.
At last, the `map_` takes only `f`, and returns a function that takes an `op`, which in turn returns
a function that takes `(acc,s)` and returns a value of the same type as `acc`.

We can think of `op` as the binary multiplication in a monoid. Hence, `op::A×A -> A`.
If this is the case, then it requires that `f:B->A`. Not that the domain of `f` can be anything, as
long as the codomain is a value that matches the type of `op`.

Our `map_` takes a value of type $A^B \times A^{(A \times A )}$, and returns $A^{(A\times A)}$.

The `map_` takes a value of type $A^B$ and rerturns a function that takes a values of type
$A^{(A \times A )}$ and returns values of type  $A^{(A \times A )}$.

In [9]:
function id_(f)
    op-> begin
        (acc,s) -> op(acc,s)
    end
end
foldl(id_(inc)(vcat), [1,2,3], init=[])

3-element Vector{Any}:
 1
 2
 3

In [10]:
function filter_(predicate)
    op-> begin
        (acc,s) -> begin
            if predicate(s)
                return op(acc,s)
            end
            return acc
        end
    end
end
foldl(filter_(x->x<2)(vcat), [1,2,3], init=[])

1-element Vector{Any}:
 1

The neat thing is that we can now compose such functions.

In [11]:
foldl(filter_(isodd)(vcat),[1,2,3,4,5], init=[])

3-element Vector{Any}:
 1
 3
 5

In [12]:
foldl(
    (map_(inc)∘filter_(isodd))(vcat),
    [1,2,3,4,5], init=[])

2-element Vector{Any}:
 3
 5

In [13]:
foldl(
    vcat |> filter_(isodd) |> map_(inc),
    [1,2,3,4,5], init=[])

2-element Vector{Any}:
 3
 5

In [14]:
foldl(filter_(isodd)(+),[1,2,3,4,5], init=0) # Filter and sum them

9

In the Glossary for the documentation of `Transducers.jl`, it reads:

This fold is distinct from the common `foldl`. The `step` is our `vcat`, while the `xf` is our composition of functions.

In [15]:
foldl(id_(isodd)(+),[1,2,3,4,5], init=0) # sum everything

15

In [16]:
function accum_(ac)
    op-> begin
        (acc,s) -> begin
            return op(acc,foldr(ac,acc,init=s))
        end
    end
end
function pair_(ac)
    op-> begin
        (acc,s) -> begin
            return op(acc,s)
        end
    end
end

# foldl(accum_(*)(vcat), [1,2,3,1,1,1], init=[])
foldl(pair_(-)(vcat), [1,2,3], init=[])

# scan_(ac) = map_(x->last(x,2)) ∘ accum_(ac)
# foldl(vcat |> accum_(*) |> map_(x->last(x,2)), [1,2,3,1,1], init=[])
# foldl(vcat |> map_(x->first(x,2)) |> accum_(*), [1,2,3,1,1], init=[])

3-element Vector{Any}:
 1
 2
 3

In [17]:
last([1,2,3],2)

2-element Vector{Int64}:
 2
 3

In [20]:
using Transducers

In [21]:
[1,2,3,1,1,1] |> Scan(*) |> collect

6-element Vector{Int64}:
 1
 2
 6
 6
 6
 6

In [22]:
foldl(id_(isodd)(+),(1,2,3,4,5), init=0) # sum everything

15

To use a transducer, we need a monoid that "accumulates", and a value which is iterable.

In [23]:
struct Ee
    z
end
function Base.iterate(v::Ee)
    return (v.z, nothing)    
end
function Base.iterate(v::Ee,state)
    return nothing
end

v = Ee(1)

foldl(id_(isodd)(+),Ee(10), init=0) # sum everything

10

In [24]:
struct E
    z
    x
    y
end
function Base.iterate(v::E)
    return (v.z, :x)    
end
function Base.iterate(v::E,state)
    if isnothing(state)
        return nothing
    elseif state == :y
        return (v.y, nothing)
    end
    return (v.x, :y)
end

v = E(1,2,3)

foldl(id_(isodd)(vcat),E(2,1,1), init=[]) # sum everything

3-element Vector{Any}:
 2
 1
 1